# Module 7.3 — Memory: Short- and Long-Term

**Goal:** Make DeskMate coherent across multi-turn conversations. Short-term memory holds the current exchange; long-term memory persists customer facts. Summarisation keeps the context window from overflowing.

## 1. Setup

In [ ]:
import os, re, json, time, dataclasses
from typing import Optional
import matplotlib.pyplot as plt
import numpy as np

SMOKE_TEST        = True
MAX_WINDOW_TURNS  = 6       # keep last 6 exchanges verbatim
SUMMARY_THRESHOLD = 1500    # approx tokens before summarisation triggers

os.makedirs('reports', exist_ok=True)
print('Config OK')

## 2. ConversationMemory — Sliding Window

In [ ]:
class ConversationMemory:
    def __init__(self, max_turns: int = MAX_WINDOW_TURNS):
        self.max_turns = max_turns
        self.messages: list[dict] = []
        self.summaries: list[str] = []  # compressed earlier turns

    def add(self, role: str, content: str):
        self.messages.append({'role': role, 'content': content})

    def token_estimate(self) -> int:
        return int(sum(len(m['content'].split()) for m in self.messages) * 1.3)

    def get_window(self) -> list[dict]:
        return self.messages[-(self.max_turns * 2):]

    def get_full_prompt_messages(self, system: str) -> list[dict]:
        msgs = [{'role': 'system', 'content': system}]
        if self.summaries:
            summary_block = '[Earlier conversation summary]\n' + '\n'.join(self.summaries)
            msgs.append({'role': 'system', 'content': summary_block})
        msgs.extend(self.get_window())
        return msgs

    def clear(self):
        self.messages = []
        self.summaries = []

# Test: 12 turns, sliding window returns last 6
mem = ConversationMemory(max_turns=6)
for i in range(12):
    mem.add('user' if i % 2 == 0 else 'assistant', f'message {i}')

window = mem.get_window()
print(f'Total messages: {len(mem.messages)}')
print(f'Window size:    {len(window)}')
print(f'Window starts at: {window[0]["content"]}')
print(f'Window ends at:   {window[-1]["content"]}')
print(f'Approx tokens:    {mem.token_estimate()}')

## 3. Summarise-and-Replace

In [ ]:
def summarise_stub(messages: list[dict]) -> str:
    lines = [m['content'] for m in messages]
    return ('Summary: ' + ' | '.join(lines[:3]) + ' ...')

def maybe_summarise(mem: ConversationMemory, llm_summarise_fn) -> bool:
    if mem.token_estimate() < SUMMARY_THRESHOLD:
        return False
    half = len(mem.messages) // 2
    to_compress = mem.messages[:half]
    summary_text = llm_summarise_fn(to_compress)
    mem.summaries.append(summary_text)
    mem.messages = mem.messages[half:]
    return True

# Test summarisation trigger
mem2 = ConversationMemory(max_turns=20)
long_content = 'word ' * 50
for i in range(30):
    mem2.add('user' if i % 2 == 0 else 'assistant', long_content)

print(f'Before summarise: {len(mem2.messages)} messages, ~{mem2.token_estimate()} tokens')
fired = maybe_summarise(mem2, summarise_stub)
print(f'Summarisation fired: {fired}')
print(f'After summarise:  {len(mem2.messages)} messages, {len(mem2.summaries)} summary block(s)')
print(f'Summary text: {mem2.summaries[0][:80]}...')

## 4. Long-Term Memory — CustomerProfile

In [ ]:
@dataclasses.dataclass
class CustomerProfile:
    customer_id:      str
    tier:             str = 'free'
    known_issues:     list = dataclasses.field(default_factory=list)
    past_resolutions: list = dataclasses.field(default_factory=list)
    preferred_lang:   str = 'en'
    last_contact:     str = ''

PROFILE_DB: dict[str, CustomerProfile] = {}

def load_profile(customer_id: str) -> Optional[CustomerProfile]:
    return PROFILE_DB.get(customer_id)

def update_profile(customer_id: str, patch: dict) -> None:
    if customer_id not in PROFILE_DB:
        PROFILE_DB[customer_id] = CustomerProfile(customer_id=customer_id)
    for k, v in patch.items():
        setattr(PROFILE_DB[customer_id], k, v)

def build_system_with_memory(profile: Optional[CustomerProfile]) -> str:
    base = (
        'You are DeskMate, a support assistant. '
        'Answer using ONLY the provided context. Cite with [Source N]. '
        'If the context does not contain the answer, escalate.'
    )
    if profile is None:
        return base
    facts = [f'Customer tier: {profile.tier}',
             f'Language: {profile.preferred_lang}']
    if profile.known_issues:
        facts.append('Open issues: ' + '; '.join(profile.known_issues))
    if profile.past_resolutions:
        facts.append('Past fixes applied: ' + '; '.join(profile.past_resolutions))
    return base + '\n\n[Customer context]\n' + '\n'.join(facts)

# Seed a profile
update_profile('cust-001', {
    'tier': 'pro',
    'known_issues': ['CSV export ERR-500 — filed BUG-0042'],
    'past_resolutions': ['upgraded to v4.3.0'],
    'last_contact': '2026-06-24',
})

profile = load_profile('cust-001')
print('Profile:', profile)
print()
print('System message with memory:')
print(build_system_with_memory(profile))

## 5. LLM and Retriever Stubs

In [ ]:
CHUNKS_DB = [
    {'source': 'release_notes.md',
     'text':   'Version 4.3.0 fixes the CSV export ERR-500 crash on all platforms.'},
    {'source': 'billing_refunds.md',
     'text':   'Billing disputes: contact support with invoice numbers within 60 days.'},
    {'source': 'shipping_policy.md',
     'text':   'Standard shipping takes 3-5 business days after dispatch.'},
]

def full_retrieve(query: str, n: int = 3) -> list:
    q = query.lower()
    if 'csv' in q or 'export' in q:
        return [CHUNKS_DB[0]]
    if 'billing' in q or 'charge' in q or 'refund' in q:
        return [CHUNKS_DB[1]]
    return CHUNKS_DB[:n]

def vllm_generate_stub(messages: list) -> str:
    history = ' '.join(m.get('content','') or '' for m in messages).lower()
    if 'already tried' in history or 'still broken' in history or 'did not help' in history:
        return 'Since the upgrade did not resolve the issue, I am escalating to our engineering team. [Source 1]'
    if 'csv' in history or 'export' in history:
        return 'Please upgrade to version 4.3.0 to fix the CSV export crash. [Source 1]'
    if 'shipping' in history or 'order' in history:
        return 'Standard shipping takes 3-5 business days after dispatch. [Source 3]'
    if 'billing' in history or 'charge' in history:
        return 'Contact support with your invoice numbers to resolve billing disputes. [Source 2]'
    return 'Thank you for contacting DeskMate support. How can I help you further?'

print('Stubs ready.')

## 6. DeskMateSession — Memory-Aware Orchestrator

In [ ]:
class DeskMateSession:
    def __init__(self, customer_id: str):
        self.customer_id = customer_id
        self.conv = ConversationMemory(max_turns=MAX_WINDOW_TURNS)
        self.profile = load_profile(customer_id)
        self.turn_count = 0

    def handle_turn(self, user_message: str) -> str:
        self.turn_count += 1

        # Add user message to short-term memory
        self.conv.add('user', user_message)

        # Proactive summarisation if approaching token limit
        maybe_summarise(self.conv, summarise_stub)

        # Build system message with long-term memory
        system = build_system_with_memory(self.profile)

        # Retrieve relevant chunks
        chunks = full_retrieve(user_message)
        ctx_block = '\n\n'.join(
            f"[Source {i+1}] {c['source']}\n{c['text']}" for i, c in enumerate(chunks))

        # Build prompt: system + summaries + window (RAG injected into last user msg)
        msgs = [{'role': 'system', 'content': system}]
        if self.conv.summaries:
            msgs.append({'role': 'system',
                         'content': '[Earlier conversation summary]\n'
                         + '\n'.join(self.conv.summaries)})
        window = self.conv.get_window()
        for i, msg in enumerate(window):
            if i == len(window) - 1 and msg['role'] == 'user':
                msgs.append({'role': 'user',
                             'content': f'Context:\n{ctx_block}\n\n{msg["content"]}'})
            else:
                msgs.append(msg)

        # Generate
        reply = vllm_generate_stub(msgs)

        # Add reply to short-term memory
        self.conv.add('assistant', reply)

        # Update long-term memory on resolution
        resolution_words = ['resolved', 'fixed', 'escalating', 'upgraded']
        if any(w in reply.lower() for w in resolution_words):
            current = self.profile.past_resolutions if self.profile else []
            update_profile(self.customer_id,
                          {'past_resolutions': current + [user_message[:80]],
                           'last_contact': '2026-06-26'})
            self.profile = load_profile(self.customer_id)

        return reply

print('DeskMateSession defined.')

## 7. Scenario 1 — CSV Export (5 Turns)

In [ ]:
print('=== Scenario 1: CSV Export ===\n')
session1 = DeskMateSession('cust-001')

conversation = [
    'My CSV export is broken. I get error ERR-500.',
    'I already tried upgrading to v4.3.0, it is still broken.',
    'I am on Windows 11, using Chrome 125.',
    'The error happens only when I export more than 1000 rows.',
    'Can you escalate this to your engineering team?',
]

for turn, user_msg in enumerate(conversation, 1):
    reply = session1.handle_turn(user_msg)
    print(f'Turn {turn}')
    print(f'  User:  {user_msg}')
    print(f'  Agent: {reply}')
    print()

print(f'Short-term memory size: {len(session1.conv.messages)} messages')
print(f'Summaries: {len(session1.conv.summaries)}')
print(f'Updated profile resolutions: {load_profile("cust-001").past_resolutions}')

## 8. Scenario 2 — Order Inquiry (4 Turns)

In [ ]:
print('=== Scenario 2: Order Inquiry ===\n')
update_profile('cust-002', {'tier': 'free', 'preferred_lang': 'en'})
session2 = DeskMateSession('cust-002')

conversation2 = [
    'Hi, I placed an order three days ago.',
    'Is standard shipping included with my plan?',
    'How many days does shipping usually take?',
    'OK, so it should arrive by Friday then?',
]

for turn, user_msg in enumerate(conversation2, 1):
    reply = session2.handle_turn(user_msg)
    print(f'Turn {turn}')
    print(f'  User:  {user_msg}')
    print(f'  Agent: {reply}')
    print()

## 9. Context Window Guard — 20 Turns

In [ ]:
print('=== Context window guard test ===\n')
session3 = DeskMateSession('cust-003')
update_profile('cust-003', {'tier': 'enterprise'})

# Simulate 20 turns with long messages
for i in range(20):
    msg = f'Turn {i}: This is a long support message about billing and CSV export issues. ' * 5
    session3.handle_turn(msg)

print(f'After 20 turns:')
print(f'  Short-term messages in window: {len(session3.conv.messages)}')
print(f'  Summary blocks:                {len(session3.conv.summaries)}')
print(f'  Approx tokens in window:       {session3.conv.token_estimate()}')
overflow_prevented = session3.conv.token_estimate() < SUMMARY_THRESHOLD * 2
print(f'  Context overflow prevented:    {overflow_prevented}')

## 10. Quality Check — With Memory vs Stateless

In [ ]:
from rouge_score import rouge_scorer as rs_module
_scorer = rs_module.RougeScorer(['rougeL'], use_stemmer=True)

# Multi-turn test: stateless model cannot use prior context
MULTI_TURN_GOLD = [
    {
        'turns': [
            'My CSV export is broken.',
            'I already tried upgrading to v4.3.0, it is still broken.',
        ],
        'ref': 'Since the upgrade did not resolve the issue, I am escalating to our engineering team.',
        'label': 'csv_post_upgrade',
    },
    {
        'turns': [
            'How many days does shipping take?',
            'Is that business days or calendar days?',
        ],
        'ref': 'Standard shipping takes 3-5 business days after dispatch.',
        'label': 'shipping_clarification',
    },
]

memory_rouges, stateless_rouges = [], []

for scenario in MULTI_TURN_GOLD:
    # With memory
    session_mem = DeskMateSession('cust-eval')
    update_profile('cust-eval', {'tier': 'pro'})
    last_reply_mem = ''
    for turn in scenario['turns']:
        last_reply_mem = session_mem.handle_turn(turn)

    # Stateless (fresh session per turn, no history)
    last_reply_nonem = ''
    for turn in scenario['turns']:
        session_nonem = DeskMateSession('cust-eval-nonem')
        last_reply_nonem = session_nonem.handle_turn(turn)

    r_mem  = _scorer.score(scenario['ref'], last_reply_mem)['rougeL'].fmeasure
    r_stat = _scorer.score(scenario['ref'], last_reply_nonem)['rougeL'].fmeasure
    memory_rouges.append(round(r_mem, 3))
    stateless_rouges.append(round(r_stat, 3))
    print(f"Scenario: {scenario['label']}")
    print(f"  Memory ROUGE-L:    {r_mem:.3f}")
    print(f"  Stateless ROUGE-L: {r_stat:.3f}  (delta {r_mem-r_stat:+.3f})")
    print()

avg_mem  = round(sum(memory_rouges) / len(memory_rouges), 3)
avg_stat = round(sum(stateless_rouges) / len(stateless_rouges), 3)
print(f'Mean memory ROUGE-L:    {avg_mem}')
print(f'Mean stateless ROUGE-L: {avg_stat}')
print(f'Mean delta:             {round(avg_mem - avg_stat, 3):+}')

## 11. Memory vs Stateless Chart

In [ ]:
labels = [s['label'] for s in MULTI_TURN_GOLD]
x = np.arange(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, memory_rouges,    w, label='With memory',    color='steelblue')
ax.bar(x + w/2, stateless_rouges, w, label='Stateless (no memory)', color='coral')
ax.set_ylabel('ROUGE-L (final turn reply)')
ax.set_title('Multi-Turn Quality: Memory vs Stateless')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=10)
ax.legend()
ax.set_ylim(0, 1.0)
for i, v in enumerate(memory_rouges):
    ax.text(i - w/2, v + 0.02, str(v), ha='center', fontsize=9)
for i, v in enumerate(stateless_rouges):
    ax.text(i + w/2, v + 0.02, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('memory_vs_stateless.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: memory_vs_stateless.png')

## 12. Checkpoint

In [ ]:
checkpoint = (
    'How do you keep a long conversation inside the context window?\n\n'
    '1. Sliding window  — keep only the last N turns verbatim (DeskMate: 6 turns).\n'
    '   Oldest messages are evicted first.\n\n'
    '2. Summarise-and-replace  — before evicting old turns, compress them into a\n'
    '   3-5 bullet summary using the LLM. The summary is injected as a system message.\n'
    '   No information is silently lost — it is compressed, not deleted.\n\n'
    '3. Proactive token budgeting  — count tokens before every API call, not after\n'
    '   a failure. When approx_tokens > SUMMARY_THRESHOLD, summarise immediately.\n\n'
    'The invariant: total prompt (system + long-term + summary + window + RAG context)\n'
    'must fit within the model context window before the call.'
)
print(checkpoint)

## 13. Save Report

In [ ]:
report = [
    '# Memory Report\n',
    f'Smoke test: {SMOKE_TEST}',
    f'MAX_WINDOW_TURNS: {MAX_WINDOW_TURNS}',
    f'SUMMARY_THRESHOLD: {SUMMARY_THRESHOLD} tokens\n',
    '## Multi-Turn ROUGE-L',
    '',
    f'Mean with memory:    {avg_mem}',
    f'Mean stateless:      {avg_stat}',
    f'Delta:               {round(avg_mem - avg_stat, 3):+}',
    '',
    '## Context Window Guard',
    '',
    f'After 20 turns — messages in window: {len(session3.conv.messages)}',
    f'Summary blocks:  {len(session3.conv.summaries)}',
    f'Overflow prevented: {overflow_prevented}',
    '',
    '## Checkpoint',
    '',
    '1. Sliding window: keep last 6 turns verbatim.',
    '2. Summarise-and-replace: compress evicted turns into a system message.',
    '3. Proactive token budgeting: count tokens before every call.',
    'Invariant: prompt must fit context window before the API call.',
]

with open('reports/memory_report.md', 'w') as f:
    f.write('\n'.join(report))

print('Saved: reports/memory_report.md')
print('\n=== Module 7.3 Complete ===')